In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import learning_curve, TimeSeriesSplit
from sklearn.metrics import r2_score, mean_absolute_error
import time, os, json, joblib

# --- config ---
INPUT_CSV  = "/Users/alexpotter/AIoT-Lab/Work/Datapreprocessing/Temp_combined_features.csv"
OUTPUT_DIR = "/Users/alexpotter/AIoT-Lab/Work/Training/Temp_Adjust/Model"
TARGET_COL = "Indoor_Temperature_C"
MULTI_STEP = 5
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Load Data ---
print("[INFO] Loading dataset...")
df = pd.read_csv(INPUT_CSV)
df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")
df = df.sort_values("Timestamp").reset_index(drop=True)
print(f"[INFO] Dataset loaded: {len(df)} rows")

# --- Create lag features ---
df["Indoor_Temp_lag1"] = df[TARGET_COL].shift(1)
df["Indoor_Temp_lag2"] = df[TARGET_COL].shift(2)

# --- Time features ---
df["Hour"] = df["Timestamp"].dt.hour
df["DayOfWeek"] = df["Timestamp"].dt.dayofweek
df["Month"] = df["Timestamp"].dt.month
df["Hour_sin"] = np.sin(2 * np.pi * df["Hour"] / 24)
df["Hour_cos"] = np.cos(2 * np.pi * df["Hour"] / 24)

# --- Multi-step targets ---
for step in range(1, MULTI_STEP+1):
    df[f"target_t+{step}"] = df[TARGET_COL].shift(-step)

# --- Drop rows with NaNs ---
required_cols = ["Indoor_Temp_lag1", "Indoor_Temp_lag2",
                 "Indoor_RH_Percent", "Outdoor_Temp_C", "Dewpoint_Temp_C",
                 "Solar_Radiation_DNI_W_m2", "HVAC_Energy_usage_KWh",
                 "Hour", "DayOfWeek", "Month", "Hour_sin", "Hour_cos"] + \
                [f"target_t+{step}" for step in range(1, MULTI_STEP+1)]
df = df.dropna(subset=required_cols)
print(f"[INFO] Dataset after dropping NaNs: {len(df)} rows")

# --- Features & targets ---
feature_cols = ["Indoor_RH_Percent", "Outdoor_Temp_C", "Dewpoint_Temp_C",
                "Solar_Radiation_DNI_W_m2", "HVAC_Energy_usage_KWh",
                "Indoor_Temp_lag1", "Indoor_Temp_lag2",
                "Hour", "DayOfWeek", "Month", "Hour_sin", "Hour_cos"]
X = df[feature_cols].values
y = df[[f"target_t+{step}" for step in range(1, MULTI_STEP+1)]].values.astype(float)

# --- Train/Test Split ---
split_idx = int(len(X)*0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]
print(f"[INFO] Training samples: {len(X_train)}, Testing samples: {len(X_test)}")


[INFO] Loading dataset...
[INFO] Dataset loaded: 455397 rows
[INFO] Dataset after dropping NaNs: 455390 rows
[INFO] Training samples: 364312, Testing samples: 91078


In [3]:

# --- Multi-output Random Forest ---
rf = RandomForestRegressor(
    n_estimators=50,     # smaller for speed
    max_depth=8,
    min_samples_split=10,
    random_state=42,
    n_jobs=-1,
    verbose=1
)
model = MultiOutputRegressor(rf)

print("[INFO] Training Random Forest for 5-step forecast...")
t0 = time.time()
model.fit(X_train, y_train)
t1 = time.time()
print(f"[INFO] Training completed in {t1 - t0:.2f}s")

# --- Predictions ---
print("[INFO] Predicting test set...")
y_pred = model.predict(X_test)

# --- Metrics per step ---
metrics = {}
for step in range(MULTI_STEP):
    metrics[f"step_{step+1}"] = {
        "MAE": float(mean_absolute_error(y_test[:, step], y_pred[:, step])),
        "R2": float(r2_score(y_test[:, step], y_pred[:, step]))
    }
print(f"[INFO] {MULTI_STEP}-step forecast metrics:", metrics)

# --- Save model & metrics ---
joblib.dump(model, os.path.join(OUTPUT_DIR, "rf_5step_no_overfit.joblib"))
with open(os.path.join(OUTPUT_DIR, "metrics_rf_5step_no_overfit.json"), "w") as f:
    import json
    json.dump(metrics, f, indent=2)
print("[INFO] Model and metrics saved.")

# --- Gradio demo function ---
def plot_prediction(start_idx=0, num_points=50):
    start_idx = int(start_idx)
    num_points = int(num_points)
    
    X_slice = X[start_idx:start_idx+num_points]
    y_true = df[TARGET_COL].values[start_idx:start_idx+num_points]
    
    y_pred_multi = model.predict(X_slice)
    
    plt.figure(figsize=(10,4))
    plt.plot(y_true, marker="o", label="Actual Temp")
    
    # Plot 5-step predictions (just first step per point for simplicity)
    plt.plot(y_pred_multi[:,0], marker="x", label="Predicted 1-step")
    
    plt.xlabel("Sample index")
    plt.ylabel("Indoor Temperature (C)")
    plt.title("Random Forest 5-Step Forecast Demo")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    return plt


[INFO] Training Random Forest for 5-step forecast...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   20.3s
[Parallel(n_jobs=-1)]: Done  50 out of  50 | elapsed:   26.6s finished
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   22.0s
[Parallel(n_jobs=-1)]: Done  50 out of  50 | elapsed:   29.2s finished
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   21.2s
[Parallel(n_jobs=-1)]: Done  50 out of  50 | elapsed:   27.5s finished
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   20.8s
[Parallel(n_jobs=-1)]: Done  50 out of  50 | elapsed:   27.1s finished
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   21.7s
[

[INFO] Training completed in 139.40s
[INFO] Predicting test set...


[Parallel(n_jobs=8)]: Done  50 out of  50 | elapsed:    0.1s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.1s
[Parallel(n_jobs=8)]: Done  50 out of  50 | elapsed:    0.1s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.1s
[Parallel(n_jobs=8)]: Done  50 out of  50 | elapsed:    0.1s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.1s
[Parallel(n_jobs=8)]: Done  50 out of  50 | elapsed:    0.1s finished


[INFO] 5-step forecast metrics: {'step_1': {'MAE': 0.02579734093315497, 'R2': 0.9994260313987099}, 'step_2': {'MAE': 0.03489967929397931, 'R2': 0.9988859616811251}, 'step_3': {'MAE': 0.04436053682451984, 'R2': 0.9981759061913008}, 'step_4': {'MAE': 0.0542242518987854, 'R2': 0.9972690287333879}, 'step_5': {'MAE': 0.06276505815873316, 'R2': 0.9963064424601473}}
[INFO] Model and metrics saved.


In [4]:
def plot_prediction_all_steps(start_idx=0, num_points=50):
    start_idx = int(start_idx)
    num_points = int(num_points)
    
    X_slice = X[start_idx:start_idx+num_points]
    y_true = df[TARGET_COL].values[start_idx:start_idx+num_points]
    
    y_pred_multi = model.predict(X_slice)
    
    plt.figure(figsize=(10,4))
    plt.plot(y_true, marker="o", label="Actual Temp")
    
    for step in range(MULTI_STEP):
        plt.plot(y_pred_multi[:,step], marker="x", label=f"Pred t+{step+1}")
    
    plt.xlabel("Sample index")
    plt.ylabel("Indoor Temperature (C)")
    plt.title("Random Forest 5-Step Forecast")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    return plt

    # --- Launch Gradio demo ---
demo = gr.Interface(
    fn=plot_prediction,
    inputs=[
        gr.Number(label="Start Index", value=0, precision=0),
        gr.Number(label="Number of Points", value=50, precision=0)
    ],
    outputs=gr.Plot(label="Temperature Prediction vs Actual"),
    live=False,
    title="Indoor Temperature Prediction Demo",
    description="Shows actual vs predicted indoor temperature using the trained Random Forest 5-step model (no overfitting)."
)

demo.launch(inline=True)


NameError: name 'gr' is not defined